In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, DateType, DecimalType, StringType

# Cargar las tablas originales (raw) para aplicar la limpieza
df_presupuestos = spark.table("presupuestos")
df_presupuesto_detalle = spark.table("presupuesto_detalle")
df_clientes = spark.table("clientes")
df_factura_cabecera = spark.table("factura_cabecera")
df_factura_detalle = spark.table("factura_detalle")
df_pago_proveedores = spark.table("pago_proveedores")
df_proveedores = spark.table("proveedores")
df_servicios = spark.table("servicios")
df_usuarios = spark.table("usuarios")




In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, datediff, to_date

# 1) Validar estructura
df_presupuestos.printSchema()
df_presupuesto_detalle.printSchema()

# 2) Unir presupuesto con su detalle
join_df = df_presupuestos.join(
    df_presupuesto_detalle,
    on="IdPresupuesto",
    how="left"
)

# 3) Columnas derivadas a nivel de fila de detalle
#    SubtotalDet = Cantidad * PrecioUnitario
join_df = join_df.withColumn("SubtotalDet", expr("Cantidad * PrecioUnitario"))

# 4) Totales por presupuesto (sumar SubtotalDet)
totales = (
    join_df.groupBy("IdPresupuesto")
           .agg(F.sum("SubtotalDet").alias("TotalCalculado"))
)
fe_dt = F.coalesce(
    expr("try_to_date(FechaEntrega, 'yyyy-MM-dd')"),
    expr("try_to_date(FechaEntrega, 'dd/MM/yyyy')"),
    expr("try_to_date(FechaEntrega, 'dd-MM-yyyy')"),
    expr("try_to_date(FechaEntrega, 'MM/dd/yyyy')")
)

fc_dt = F.coalesce(
    expr("try_to_date(FechaCrea, 'yyyy-MM-dd')"),
    expr("try_to_date(FechaCrea, 'dd/MM/yyyy')"),
    expr("try_to_date(FechaCrea, 'dd-MM-yyyy')"),
    expr("try_to_date(FechaCrea, 'MM/dd/yyyy')")
)

dfp = (
    df_presupuestos
      .withColumn("FechaEntrega_dt", fe_dt)
      .withColumn("FechaCrea_dt",    fc_dt)
)

# ❗ Filtra filas que NO tienen ambas fechas válidas (descarta 'STO PEZ', 'LIMA', '0', etc.)
dfp_valid = dfp.filter(F.col("FechaEntrega_dt").isNotNull() & F.col("FechaCrea_dt").isNotNull())

# join_final usando sólo filas válidas y calculando la duración
join_final = (
    dfp_valid
      .join(totales, on="IdPresupuesto", how="left")
      .fillna({"TotalCalculado": 0.0})
      .withColumn("DiferenciaTotal", col("Total") - col("TotalCalculado"))
      .withColumn("DuracionEntrega", F.datediff(F.col("FechaEntrega_dt"), F.col("FechaCrea_dt")))
)


display(join_df)
display(totales)
display(join_final)

root
 |-- IdPresupuesto: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- IdCliente: long (nullable = true)
 |-- Proyecto: string (nullable = true)
 |-- Lugar: string (nullable = true)
 |-- FechaEntrega: string (nullable = true)
 |-- TipoEvento: string (nullable = true)
 |-- Motivo: long (nullable = true)
 |-- Detalle: string (nullable = true)
 |-- IdEmpleado: string (nullable = true)
 |-- IdEstadoPresupuesto: string (nullable = true)
 |-- IdEjecutivo: string (nullable = true)
 |-- SubTotal: string (nullable = true)
 |-- Igv: string (nullable = true)
 |-- Total: double (nullable = true)
 |-- IdUsuario: double (nullable = true)
 |-- FechaCrea: string (nullable = true)
 |-- SaldoInicial: string (nullable = true)
 |-- SaldoActual: string (nullable = true)
 |-- SaldoFinal: string (nullable = true)
 |-- OrdenDeCompra: string (nullable = true)
 |-- TipoFacturacion: string (nullable = true)
 |-- Interes: double (nullable = true)
 |-- Liquidado: long (nullable = true)
 |-- Liquid

IdPresupuesto,Fecha,IdCliente,Proyecto,Lugar,FechaEntrega,TipoEvento,Motivo,Detalle,IdEmpleado,IdEstadoPresupuesto,IdEjecutivo,SubTotal,Igv,Total,IdUsuario,FechaCrea,SaldoInicial,SaldoActual,SaldoFinal,OrdenDeCompra,TipoFacturacion,Interes,Liquidado,LiquidadoTotal,AInterno,ColCotizacion,Estado,IdPresupuestoDet,IdServicio,Item,IdRubro,DetalleServicio,Cantidad,IdUnidadMedida,Fechas,PrecioUnitario,SubTotal,ItemRubro,DescripcionRubro,DescripcionServicio,SubtotalDet
92,2024-07-17,86,IMPLEMENTACION PLANTA PIURA,HUACHIPA,2024-07-22,0,0,CABINA CON AIRE ACONDICIONADO PARA ESCOBAS,null,null,0,22,3.96,25.96,1076.0,6/07/2023,null,null,null,0000049815,null,null,null,null,null,null,269,26058,2030,1,155,40 mm de espesor,120,0,1,185.0,22.0,1,PANELERIA Y SOPORTES,PANELES DE PARED PIR,22200.0
195,2024-10-02,81,GRABACION - REGISTRO,null,2024-11-04,0,0,null,null,null,0,1069.65,192.54,1262.19,33.0,5/09/2023,null,null,null,3334,1,null,null,null,null,null,244,28780,566,5,62,Comisión de agencia y costo financiero a 30 días.,1,0,1,1069.65,1069.65,2,HONORARIOS,FEE DE AGENCIA,1069.65
212,2024-07-19,8,CINEMARK 2024,STAND COMETA FEST,2024-07-25,0,0,null,null,null,0,10110,1819.8,11929.8,33.0,20/09/2023,null,null,null,201001940,1,5.0,null,1,null,0,269,28984,581,13,51,null,1,0,1,1470.0,1470.0,5,FEE DE AGENCIA,HONORARIOS - ORGANIZACIÓN - SUPERVISIÓN,1470.0
274,2024-08-22,81,RED 2024 - CUMBRE DE LIDERES,PLANTA CALLAO,2024-08-28,176,239,TRABAJO INTERMEDIARIO VIRTUAL,null,null,0,8900,1602,10502.0,33.0,19/01/2024,null,null,null,1111111,1,5.0,null,1,null,null,269,22511,2114,1,160,"Fecha: 28 de agosto Locación: Flota Callao Internet: En la sede el cliente deberá proveer un punto de Internet Inversión: S/. 8, 900.00 más I.G.V. (Ocho mil seiscientos. NO incluye IGV). ** Pago 30 días Visita Técnica: 1. Instalación de pantalla un día antes. 2. Ensayo técnico un día antes Detalles técnicos del servicio de 1 pantallas LED y Equipo de sonido. Para las Sede de Planta Callao • Pantalla led de 4 x 2.5 mts (P3) • Estructura para pantalla • 2 parlantes JBL PRX 815 • 02 micrófonos inalámbricos Shure blx sm58 • Técnico a cargo - personal de montaje • Registro audiovisual - video resumen - entrevistas • Transporte El cliente deberá proporcionar: Punto de energía estabilizado trifásico. Detalles técnicos de producción para 06 personas: • Exámenes OIS • Exámenes EMO • Prevencionistas de riesgos (2 días) • Desarrollo de matrices • Movilidad • Productor general • Gastos de pre Producción • Viáticos personales Teléfono:",1,0,1,8900.0,8900.0,1,TOTAL - INVERSIÓN,RED EN RUTA - CUMBRE DE LIDERES,8900.0
289,2024-10-25,1,PASEO DE LA BUENA FAMILIA - PAVIFERIA 2024,VARIOS,2024-12-21,261,0,BTL - IMPLEMENTACION,null,null,39,202159,36388.62,238547.62,33.0,7/02/2024,null,null,null,4501732383,1,0.0,null,1,b6d0e49a-c01c-424e-99e3-6d1595802a11.pdf,null,269,28400,1876,1,1,6 PUNTOS,1,0,1,202159.0,202159.0,1,PRE-PRODUCCION,LOCACIÓN,202159.0
295,2024-08-26,85,PROYECTO 300 MTS ESTRUCTURA,PLANTA HUACHIPA,2024-09-15,0,0,null,null,null,0,13000,2340,15340.0,1076.0,13/02/2024,null,null,null,null,null,null,null,null,null,null,269,25257,2159,1,169,300 mts de estructura de acero,1,0,1,13000.0,13000.0,1,ESTRUCTURA,CONSTRUCCION ESTRUCTURA,13000.0
302,2025-02-02,62,PERUMIN 2025,AREQUIPA,null,0,0,null,null,null,0,79251,14265.18,93516.18,33.0,21/02/2024,null,null,null,null,null,null,null,null,null,0,117,31000,2306,29,185,null,1,0,1,78000.0,78000.0,7,VALOR TOTAL DEL STAND,Monto alquiler,78000.0
309,2024-03-04,52,STAND - PRODIMIN,STAND,null,0,0,null,null,null,0,4085,735.3,4820.3,33.0,4/03/2024,null,null,null,OCN20240069,1,5.0,null,1,null,null,269,17707,581,6,51,null,1,0,1,800.0,800.0,2,FEE DE AGENCIA,HONORARIOS - ORGANIZACIÓN - SUPERVISIÓN,800.0
315,2024-05-28,52,COCTEL PARTNERS,STO PEZ,2024-02-20,0,0,null,null,null,0,17300,3114,20414.0,33.0,6/03/2024,null,null,null,2024070005,1,5.0,null,1,null,null,269,20582,581,11,51,null,1,0,1,1434.0,1434.0,7,FEE DE AGENCIA,HONORARIOS - ORGANIZACIÓN - SUPERVISIÓN,1434.0
326,2024-04-

IdPresupuesto,TotalCalculado
1785,12793.18
271,null
372,null
212,9860.0
1748,48185.0
362,5359.0
331,12468.06
547,13834.5
1768,12000.0
459,30000.0


IdPresupuesto,Fecha,IdCliente,Proyecto,Lugar,FechaEntrega,TipoEvento,Motivo,Detalle,IdEmpleado,IdEstadoPresupuesto,IdEjecutivo,SubTotal,Igv,Total,IdUsuario,FechaCrea,SaldoInicial,SaldoActual,SaldoFinal,OrdenDeCompra,TipoFacturacion,Interes,Liquidado,LiquidadoTotal,AInterno,ColCotizacion,Estado,FechaEntrega_dt,FechaCrea_dt,TotalCalculado,DiferenciaTotal,DuracionEntrega
29,2023-03-28,15,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,2023-04-06,21,0,MAKRO SJL 2,null,null,0,18482.2,3326.8,21809.0,75.0,28/03/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-03-28,0.0,21809.0,9
48,2023-04-11,49,REGISTRO AUDIOVISUAL - 24 DE ABRIL,REGISTRO AUDIOVISUAL - 24 DE ABRIL,2023-04-24,20,0,REGISTRO AUDIOVISUAL DE CONFERENCIA,null,null,8,0,0,0.0,33.0,11/04/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-24,2023-04-11,0.0,0.0,13
49,2023-04-06,15,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,2023-04-06,21,0,MAKRO HUAYLAS,null,null,0,21809,3925.62,25734.62,75.0,29/04/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-04-29,0.0,25734.62,-23
51,2023-05-16,50,Festival Intercorp Puruchuco,Festival Intercorp Puruchuco,2023-07-01,21,1,Parecido a Caravana de Abril,null,null,10,16646,2996.28,19642.28,33.0,16/05/2023,null,null,null,null,null,null,null,null,null,null,269,2023-07-01,2023-05-16,0.0,19642.28,46
56,2023-05-28,40,CARAVANA INTERCORP LIMA ESTE - PURUCHUCO,CARAVANA INTERCORP LIMA ESTE - PURUCHUCO,2023-07-01,0,0,Feria de Stands del Gpo Intercorp,null,null,11,3108.91,559.6,3668.51,33.0,28/05/2023,null,null,null,4400585978,1,null,null,1,null,null,269,2023-07-01,2023-05-28,0.0,3668.51,34
73,2023-06-16,15,Participación Notarial - Mejores Clientes Mayo,Participación Notarial - Mejores Clientes Mayo,2023-04-06,21,0,Apoyo producción Mejores clientes Makro - Premiación ganadores Mayo,0,0,14,1046.25,188.32,1234.58,75.0,16/06/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-06-16,0.0,1234.58,-71
75,2023-06-20,15,NOCHE DE ALIADOS CUZCO,NOCHE DE ALIADOS CUZCO,2023-06-15,0,0,null,null,null,0,32011.88,5762.14,37774.02,75.0,20/06/2023,null,null,null,4400594220,null,null,null,null,null,null,269,2023-06-15,2023-06-20,0.0,37774.02,-5
76,2023-06-20,15,NOCHE DE ALIADOS HUACHO,NOCHE DE ALIADOS HUACHO,2023-06-15,0,0,null,null,null,0,30661.88,5519.14,36181.02,75.0,23/06/2023,null,null,null,4400594220,null,null,null,null,null,null,269,2023-06-15,2023-06-23,0.0,36181.02,-8
77,2023-06-23,34,CARAVANA SALAVERRY,CARAVANA SALAVERRY,2023-06-24,0,0,14 STAND EN SISTEMA OCTANORM - GRUPO INTERCORP,null,null,0,2572.14,462.99,3035.13,33.0,23/06/2023,null,null,null,4500101274,null,null,null,null,null,null,269,2023-06-24,2023-06-23,0.0,3035.13,1
78,2023-06-23,34,CARAVANA SALAVERRY,CARAVANA SALAVERRY,2023-06-24,0,0,14 STAND EN SISTEMA OCTANORM - GRUPO INTERCORP,null,null,0,2572.14,462.99,3035.13,33.0,23/06/2023,null,null,null,4500101274,1,null,null,1,null,null,269,2023-06-24,2023-06-23,0.0,3035.13,1


In [0]:
# ============================================================
# TRANSFORMACIÓN INTEGRAL DE DATOS
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, datediff, to_date

# --------------------------------------------
# 2) Unión con CLIENTES
# --------------------------------------------
presupuesto_cliente = (
    join_final.join(df_clientes, on="IdCliente", how="left")
               .withColumnRenamed("Nombre", "NombreCliente")
)

# ============================
# HELPERS
# ============================
def has_cols(df, cols):
    cols = [cols] if isinstance(cols, str) else cols
    return all(c in df.columns for c in cols)

def safe_join(left, right, on, how="left"):
    on_cols = [on] if isinstance(on, str) else on
    if not has_cols(left, on_cols) or not has_cols(right, on_cols):
        return left
    return left.join(right, on=on_cols, how=how)

# ============================
# 3) FACTURAS (normalización de llaves y unión)
# ============================

# Normaliza: si detalle trae IdFacturaCab, lo mapeamos a IdFactura
df_factura_detalle_norm = df_factura_detalle
if "IdFacturaCab" in df_factura_detalle_norm.columns and "IdFactura" not in df_factura_detalle_norm.columns:
    df_factura_detalle_norm = df_factura_detalle_norm.withColumnRenamed("IdFacturaCab", "IdFactura")

# Asegura que cabecera tenga IdFactura
df_factura_cabecera_norm = df_factura_cabecera
# (si tu cabecera se llama IdFacturaCab, renómbrala)
if "IdFacturaCab" in df_factura_cabecera_norm.columns and "IdFactura" not in df_factura_cabecera_norm.columns:
    df_factura_cabecera_norm = df_factura_cabecera_norm.withColumnRenamed("IdFacturaCab", "IdFactura")

# Total facturado por factura = SUM(Cantidad * PrecioUnitario)
facturas_total = (
    df_factura_detalle_norm.groupBy("IdFactura")
    .agg(F.sum(expr("Cantidad * PrecioUnitario")).alias("TotalFactura"))
)

# Une total con cabecera
facturas_full = safe_join(df_factura_cabecera_norm, facturas_total, on="IdFactura", how="left")

# Une facturas al presupuesto 
presupuesto_factura = presupuesto_cliente
if "IdPresupuesto" in presupuesto_cliente.columns and "IdPresupuesto" in facturas_full.columns:
    presupuesto_factura = presupuesto_cliente.join(facturas_full, on="IdPresupuesto", how="left")
# Si no existe relación por IdPresupuesto, nos quedamos con presupuesto_cliente tal cual.


# ============================
# 4) PAGOS A PROVEEDORES (normalización y unión)
# ============================

# Normaliza: Proveedores trae IdProv -> renombramos a IdProveedor
df_proveedores_renamed = (
    df_proveedores
        .withColumnRenamed("IdProv", "IdProveedor") 
        if "IdProv" in df_proveedores.columns and "IdProveedor" not in df_proveedores.columns
        else df_proveedores
)

# Normaliza: pagos puede traer IdProv
df_pago_proveedores_norm = (
    df_pago_proveedores.withColumn("IdProveedor", F.coalesce(col("IdProveedor"), col("IdProv")))
    if "IdProv" in df_pago_proveedores.columns
    else df_pago_proveedores
)

# Total de pagos por proveedor
pagos_total_by_proveedor = (
    df_pago_proveedores_norm.groupBy("IdProveedor")
    .agg(F.sum("Monto").alias("MontoPagado"))
)

pagos_total_by_presupuesto = None
if "IdPresupuesto" in df_pago_proveedores_norm.columns:
    pagos_total_by_presupuesto = (
        df_pago_proveedores_norm.groupBy("IdPresupuesto")
        .agg(F.sum("Monto").alias("MontoPagado_Presupuesto"))
    )

# Une totales de pago con proveedores
pagos_prov = safe_join(pagos_total_by_proveedor, df_proveedores_renamed, on="IdProveedor", how="left")

# 1) Si presupuesto_factura TIENE IdProveedor -> une por IdProveedor
presupuesto_pago = presupuesto_factura
if has_cols(presupuesto_factura, "IdProveedor"):
    presupuesto_pago = presupuesto_factura.join(pagos_prov, on="IdProveedor", how="left")
# 2) Si NO tiene IdProveedor pero SÍ tenemos totales por IdPresupuesto -> une por IdPresupuesto
elif pagos_total_by_presupuesto is not None and has_cols(presupuesto_factura, "IdPresupuesto"):
    presupuesto_pago = presupuesto_factura.join(pagos_total_by_presupuesto, on="IdPresupuesto", how="left") \
                                         .withColumnRenamed("MontoPagado_Presupuesto", "MontoPagado")

# Visual rápido
display(pagos_prov.limit(10))
display(presupuesto_pago.limit(10))
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, datediff, to_date

# ============================
# 5) SERVICIOS
# ============================

servicio_por_presupuesto = None
if has_cols(df_presupuesto_detalle, ["IdPresupuesto", "IdServicio"]) and has_cols(df_servicios, ["IdServicio", "TipoServicio"]):
    det_serv = df_presupuesto_detalle.join(
        df_servicios.select("IdServicio", "TipoServicio"), on="IdServicio", how="left"
    )
    # Tomamos, por ejemplo, el primer tipo de servicio por presupuesto (o el más frecuente)
    w = Window.partitionBy("IdPresupuesto").orderBy(F.col("TipoServicio"))
    servicio_por_presupuesto = det_serv.withColumn("rn", F.row_number().over(w)) \
                                       .filter(col("rn") == 1) \
                                       .select("IdPresupuesto", "TipoServicio") \
                                       .withColumnRenamed("TipoServicio", "TipoServicioPrincipal")

df_con_servicios = presupuesto_pago
if servicio_por_presupuesto is not None and has_cols(presupuesto_pago, "IdPresupuesto"):
    df_con_servicios = presupuesto_pago.join(servicio_por_presupuesto, on="IdPresupuesto", how="left")


# ============================
# 6) MÉTRICAS FINALES
# ============================
df_dataset_final = (
    df_con_servicios
        .withColumn("MontoPagado", F.coalesce(col("MontoPagado"), F.lit(0.0)))
        .withColumn("MargenEstimado", F.when(col("Total").isNotNull(), col("Total") - col("MontoPagado")).otherwise(F.lit(None)))
        .withColumn("PorcentajePagado",
                    F.when((col("Total").isNotNull()) & (col("Total") > 0), col("MontoPagado") / col("Total"))
                     .otherwise(F.lit(0.0)))
        .withColumn("EstadoProyecto",
                    F.when(col("DiferenciaTotal") == 0, "Completado")
                     .when(col("DiferenciaTotal") < 0, "Subestimado")
                     .otherwise("Pendiente"))
)

# ============================
# 7) VISUALIZACIÓN
# ============================
display(df_dataset_final.select(
    "IdPresupuesto", "RazonSocial", "Total", "TotalCalculado", "DiferenciaTotal",
    "DuracionEntrega", "MontoPagado", "MargenEstimado", "PorcentajePagado",
     "EstadoProyecto"
).limit(500))


IdProveedor,MontoPagado,Ruc,RazonSocial,Telefono,IdContacto,ExoneradoIgv,TipoComprobante,Contacto,Servicio,IdCuentaBancaria,NroCuenta,CCI,Dni,Direccion,TieneUsuario,Correo,CT,NotaLibre
372,650.0,15605207623,TOVAR MORENO LUIS CARLOS,918 111 053,null,null,null,null,null,null,null,null,null,null,null,null,null,null
212,76.0,20602722792,AL DENTE TRATTORIA,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
673,66.0,20475733518,CORP SANCHEZ Y ASOCIADOS SRL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
290,4614.0,20523074734,HOLIDAY Y PRODUCCIONES EIRL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
435,3240.0,10103824201,VICENTE MILLA NORMAN MARTIN,992698070,null,null,F,"SR, NORMAN",MOVILIDAD,0,2843345124646,00328401334512464675,10382420,COMAS,1,martinvic40@gmail.com,1,TAXY
332,340.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
240,472.0,20550267005,DAL ESTRUCTURAS SAC,null,null,null,F,null,ALQ SALA,6,1942063801073,00219400206380107394,null,null,null,null,null,null
117,1185.04,20608300393,COMPAÑIA FOOD RETAIL S.A.C,null,null,null,F,null,PLAZA VEA,0,null,null,null,null,null,null,null,null
184,100.0,20334129595,GRIFO SERVITOR S.A,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
203,472.0,20521159490,DIAXIS SAC,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


IdPresupuesto,IdCliente,Fecha,Proyecto,Lugar,FechaEntrega,TipoEvento,Motivo,Detalle,IdEmpleado,IdEstadoPresupuesto,IdEjecutivo,SubTotal,Igv,Total,IdUsuario,FechaCrea,SaldoInicial,SaldoActual,SaldoFinal,OrdenDeCompra,TipoFacturacion,Interes,Liquidado,LiquidadoTotal,AInterno,ColCotizacion,Estado,FechaEntrega_dt,FechaCrea_dt,TotalCalculado,DiferenciaTotal,DuracionEntrega,Ruc,RazonSocial,Telefono,Direccion,Distrito,Provincia,Departamento,Rubro,TipoCliente,MontoPagado
29,15,2023-03-28,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,2023-04-06,21,0,MAKRO SJL 2,null,null,0,18482.2,3326.8,21809.0,75.0,28/03/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-03-28,0.0,21809.0,9,20492092313,MAKRO SUPERMAYORISTA S.A.,null,AV. JORGE CHAVEZ NRO. 1218 LIMA - LIMA - SANTIAGO DE SURCO,SANTIAGO DE SURCO,null,null,"VTA. DE ALIMENTOS, BEBIDAS, PRODUCTOS",336,null
48,49,2023-04-11,REGISTRO AUDIOVISUAL - 24 DE ABRIL,REGISTRO AUDIOVISUAL - 24 DE ABRIL,2023-04-24,20,0,REGISTRO AUDIOVISUAL DE CONFERENCIA,null,null,8,0,0,0.0,33.0,11/04/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-24,2023-04-11,0.0,0.0,13,20507776931,OFICINA DE LAS NACIONES UNIDAS DE SERVICIOS PARA PROYECTOS,null,AV. LOS LIBERTADORES NRO. 757 URB. SANTA CRUZ LIMA - LIMA - SAN ISIDRO,null,null,null,ACT. DE ORG. Y ÓRGANOS EXTRATERRITORIALES,337,null
49,15,2023-04-06,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,NOCHE DE ALIADOS LIMA 2023 - ABRIL 2,2023-04-06,21,0,MAKRO HUAYLAS,null,null,0,21809,3925.62,25734.62,75.0,29/04/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-04-29,0.0,25734.62,-23,20492092313,MAKRO SUPERMAYORISTA S.A.,null,AV. JORGE CHAVEZ NRO. 1218 LIMA - LIMA - SANTIAGO DE SURCO,SANTIAGO DE SURCO,null,null,"VTA. DE ALIMENTOS, BEBIDAS, PRODUCTOS",336,null
51,50,2023-05-16,Festival Intercorp Puruchuco,Festival Intercorp Puruchuco,2023-07-01,21,1,Parecido a Caravana de Abril,null,null,10,16646,2996.28,19642.28,33.0,16/05/2023,null,null,null,null,null,null,null,null,null,null,269,2023-07-01,2023-05-16,0.0,19642.28,46,00000000000,Grupo Intercorp,null,null,null,null,null,null,null,null
56,40,2023-05-28,CARAVANA INTERCORP LIMA ESTE - PURUCHUCO,CARAVANA INTERCORP LIMA ESTE - PURUCHUCO,2023-07-01,0,0,Feria de Stands del Gpo Intercorp,null,null,11,3108.91,559.6,3668.51,33.0,28/05/2023,null,null,null,4400585978,1,null,null,1,null,null,269,2023-07-01,2023-05-28,0.0,3668.51,34,20511315922,REAL PLAZA S.R.L,null,AV. PUNTA DEL ESTE NRO. 2403 (PUERTA 4 - PISO 2) LIMA - LIMA - JESUS MARIA,null,null,null,SERVICIO Y VTA DE PRODUCTOS,336,3301.51
73,15,2023-06-16,Participación Notarial - Mejores Clientes Mayo,Participación Notarial - Mejores Clientes Mayo,2023-04-06,21,0,Apoyo producción Mejores clientes Makro - Premiación ganadores Mayo,0,0,14,1046.25,188.32,1234.58,75.0,16/06/2023,null,null,null,null,null,null,null,null,null,null,269,2023-04-06,2023-06-16,0.0,1234.58,-71,20492092313,MAKRO SUPERMAYORISTA S.A.,null,AV. JORGE CHAVEZ NRO. 1218 LIMA - LIMA - SANTIAGO DE SURCO,SANTIAGO DE SURCO,null,null,"VTA. DE ALIMENTOS, BEBIDAS, PRODUCTOS",336,null
75,15,2023-06-20,NOCHE DE ALIADOS CUZCO,NOCHE DE ALIADOS CUZCO,2023-06-15,0,0,null,null,null,0,32011.88,5762.14,37774.02,75.0,20/06/2023,null,null,null,4400594220,null,null,null,null,null,null,269,2023-06-15,2023-06-20,0.0,37774.02,-5,20492092313,MAKRO SUPERMAYORISTA S.A.,null,AV. JORGE CHAVEZ NRO. 1218 LIMA - LIMA - SANTIAGO DE SURCO,SANTIAGO DE SURCO,null,null,"VTA. DE ALIMENTOS, BEBIDAS, PRODUCTOS",336,null
76,15,2023-06-20,NOCHE DE ALIADOS HUACHO,NOCHE DE ALIADOS HUACHO,2023-06-15,0,0,null,null,null,0,30661.88,5519.14,36181.02,75.0,23/06/2023,null,null,null,4400594220,null,null,null,null,null,null,269,2023-06-15,2023-06-23,0.0,36181.02,-8,20492092313,MAKRO SUPERMAYORISTA S.A.,null,AV. JORGE CHAVEZ NRO. 1218 LIMA - LIMA - SANTIAGO DE SURCO,SANTIAGO DE SURCO,null,null,"VTA. DE ALIMENTOS, BEBIDAS, PRODUCTOS",336,null
77,34,2023-06-23,CARAVANA SALAVERRY,CARAVANA S

IdPresupuesto,RazonSocial,Total,TotalCalculado,DiferenciaTotal,DuracionEntrega,MontoPagado,MargenEstimado,PorcentajePagado,EstadoProyecto
29,MAKRO SUPERMAYORISTA S.A.,21809.0,0.0,21809.0,9,0.0,21809.0,0.0,Pendiente
48,OFICINA DE LAS NACIONES UNIDAS DE SERVICIOS PARA PROYECTOS,0.0,0.0,0.0,13,0.0,0.0,0.0,Completado
49,MAKRO SUPERMAYORISTA S.A.,25734.62,0.0,25734.62,-23,0.0,25734.62,0.0,Pendiente
51,Grupo Intercorp,19642.28,0.0,19642.28,46,0.0,19642.28,0.0,Pendiente
56,REAL PLAZA S.R.L,3668.51,0.0,3668.51,34,3301.51,367.0,0.8999593840551069,Pendiente
73,MAKRO SUPERMAYORISTA S.A.,1234.58,0.0,1234.58,-71,0.0,1234.58,0.0,Pendiente
75,MAKRO SUPERMAYORISTA S.A.,37774.02,0.0,37774.02,-5,0.0,37774.02,0.0,Pendiente
76,MAKRO SUPERMAYORISTA S.A.,36181.02,0.0,36181.02,-8,0.0,36181.02,0.0,Pendiente
77,TIENDAS PERUANAS S.A.,3035.13,0.0,3035.13,1,0.0,3035.13,0.0,Pendiente
78,TIENDAS PERUANAS S.A.,3035.13,0.0,3035.13,1,2572.0,463.1300000000001,0.8474101603555696,Pendiente


In [0]:
from pyspark.sql import functions as F

# 1) Detectar columnas numéricas (omitimos IDs)
numeric_types = {"double","float","int","bigint","decimal","smallint","tinyint"}
num_cols = [c for c,t in df_dataset_final.dtypes if t in numeric_types and not c.lower().startswith("id")]

# 2) Castear Decimal -> Double para evitar problemas abajo
for c,t in df_dataset_final.dtypes:
    if t.startswith("decimal"):
        df_dataset_final = df_dataset_final.withColumn(c, F.col(c).cast("double"))

# 3) Calcular medias por columna (una sola pasada)
means_row = df_dataset_final.select(*[F.mean(F.col(c)).alias(c) for c in num_cols]).collect()[0]
means = {c: means_row[c] for c in num_cols if means_row[c] is not None}

# 4) Imputar nulos con la media (solo en columnas con media calculada)
df_num = df_dataset_final.fillna(means)

print(f"Numéricas candidatas ({len(num_cols)}):", num_cols[:20], "..." if len(num_cols)>20 else "")
print(f"Columnas imputadas ({len(means)}):", list(means.keys())[:20], "..." if len(means)>20 else "")


Numéricas candidatas (13): ['Motivo', 'Total', 'Interes', 'Liquidado', 'LiquidadoTotal', 'Estado', 'TotalCalculado', 'DiferenciaTotal', 'DuracionEntrega', 'TipoCliente', 'MontoPagado', 'MargenEstimado', 'PorcentajePagado'] 
Columnas imputadas (13): ['Motivo', 'Total', 'Interes', 'Liquidado', 'LiquidadoTotal', 'Estado', 'TotalCalculado', 'DiferenciaTotal', 'DuracionEntrega', 'TipoCliente', 'MontoPagado', 'MargenEstimado', 'PorcentajePagado'] 


In [0]:
from itertools import combinations
from pyspark.sql import functions as F

# 2.1 Eliminar columnas constantes (baja varianza)
distinct_counts = df_num.select(*[F.countDistinct(F.col(c)).alias(c) for c in num_cols]).collect()[0].asDict()
low_var = [c for c,v in distinct_counts.items() if v <= 1]
df_num = df_num.drop(*low_var)
num_cols = [c for c in num_cols if c not in low_var]

# 2.2 Eliminar alta correlación (umbral conservador)
thr = 0.97
N = df_num.count()
sample = df_num.select(*num_cols).sample(False, 0.2, seed=42) if N > 250_000 else df_num.select(*num_cols)

drop_corr = set()
for a, b in combinations(num_cols, 2):
    r = sample.select(F.corr(a, b).alias("r")).first()["r"]
    if r is not None and abs(r) > thr:
        drop_corr.add(b)

df_num = df_num.drop(*drop_corr)
num_cols = [c for c in num_cols if c not in drop_corr]

print("Eliminadas por baja varianza:", low_var)
print(f"Eliminadas por correlación (>|{thr}|):", sorted(drop_corr))
print("Num_cols finales:", num_cols)


Eliminadas por baja varianza: ['Liquidado', 'LiquidadoTotal']
Eliminadas por correlación (>|0.97|): ['MargenEstimado']
Num_cols finales: ['Motivo', 'Total', 'Interes', 'Estado', 'TotalCalculado', 'DiferenciaTotal', 'DuracionEntrega', 'TipoCliente', 'MontoPagado', 'PorcentajePagado']
